<a href="https://colab.research.google.com/github/KaRyzhova/Code-1/blob/experiment/AVL_%D0%B4%D0%B5%D1%80%D0%B5%D0%B2%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Node:
    __slots__ = ("key", "left", "right", "height")

    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1


class AVLTree:

    def __init__(self):
        self.root = None

    def _height(self, node):
        return node.height if node else 0

    def _update_height(self, node):
        node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _balance_factor(self, node):
        return self._height(node.left) - self._height(node.right) if node else 0

    def _rotate_right(self, y):
        x = y.left
        t2 = x.right

        # Выполняем поворот
        x.right = y
        y.left = t2

        # Обновляем высоты (сначала y, потом x)
        self._update_height(y)
        self._update_height(x)

        return x  # Новый корень поддерева

    def _rotate_left(self, x):
        y = x.right
        t2 = y.left

        # Выполняем поворот
        y.left = x
        x.right = t2

        # Обновляем высоты (сначала x, потом y)
        self._update_height(x)
        self._update_height(y)

        return y  # Новый корень поддерева

    def _rebalance(self, node):
        self._update_height(node)
        bf = self._balance_factor(node)

        # Левый перевес
        if bf > 1:
            # Случай LR
            if self._balance_factor(node.left) < 0:
                node.left = self._rotate_left(node.left)
            # Случай LL
            return self._rotate_right(node)

        # Правый перевес
        if bf < -1:
            # Случай RL
            if self._balance_factor(node.right) > 0:
                node.right = self._rotate_right(node.right)
            # Случай RR
            return self._rotate_left(node)

        return node

    def _insert(self, node, key):
        if node is None:
            return Node(key)

        if key < node.key:
            node.left = self._insert(node.left, key)
        elif key > node.key:
            node.right = self._insert(node.right, key)
        else:
            return node

        # Балансируем на обратном пути
        return self._rebalance(node)

    def _min_value_node(self, node):
        current = node
        while current.left is not None:
            current = current.left
        return current

    def _delete(self, node, key):
        if node is None:
            return None

        if key < node.key:
            node.left = self._delete(node.left, key)
        elif key > node.key:
            node.right = self._delete(node.right, key)
        else:
            # Узел найден. Обрабатываем 3 случая:
            # Случай 1 и 2: 0 или 1 ребенок
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left

            # Случай 3: 2 ребенка
            succ = self._min_value_node(node.right)
            node.key = succ.key
            node.right = self._delete(node.right, succ.key)

        # Балансируем на обратном пути
        return self._rebalance(node)

    def _search(self, node, key):
        if node is None or node.key == key:
            return node
        if key < node.key:
            return self._search(node.left, key)
        return self._search(node.right, key)

    def _is_balanced(self, node):
        if node is None:
            return True

        if abs(self._balance_factor(node)) > 1:
            return False

        return self._is_balanced(node.left) and self._is_balanced(node.right)

    def _inorder(self, node, result):
        if node:
            self._inorder(node.left, result)
            result.append(node.key)
            self._inorder(node.right, result)
        return result

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def delete(self, key):
        self.root = self._delete(self.root, key)

    def search(self, key):
        return self._search(self.root, key)

    def height(self):
        return self._height(self.root)

    def is_balanced(self):
        return self._is_balanced(self.root)

    def inorder(self):
        return self._inorder(self.root, [])


if __name__ == "__main__":
    print("ТЕСТИРОВАНИЕ AVL ДЕРЕВА")
    avl = AVLTree()

    # 1. Тест вставки и автоматической балансировки
    print("\n1. Вставка элементов: 10, 20, 30, 40, 50, 25")
    # Эти элементы специально идут по возрастанию, чтобы вызвать дисбаланс
    for val in [10, 20, 30, 40, 50, 25]:
        avl.insert(val)

    print(f"Корень дерева: {avl.root.key}")
    print(f"Высота дерева: {avl.height()}")
    print(f"Дерево сбалансировано? {avl.is_balanced()}")
    print(f"Отсортированный обход: {avl.inorder()}")

    # 2. Тест поиска
    print("\n2. Поиск элементов:")
    print(f"Поиск 25: {'Найден' if avl.search(25) else 'Не найден'}")
    print(f"Поиск 100: {'Найден' if avl.search(100) else 'Не найден'}")

    # 3. Тест удаления и сохранения баланса
    print("\n3. Удаление элементов:")
    print("Удаляем 40...")
    avl.delete(40)
    print(f"Высота: {avl.height()}, Сбалансировано: {avl.is_balanced()}")
    print(f"Отсортированный обход: {avl.inorder()}")

    print("Удаляем 10 (корень)...")
    avl.delete(10)
    print(f"Высота: {avl.height()}, Сбалансировано: {avl.is_balanced()}")
    print(f"Отсортированный обход: {avl.inorder()}")

    # 4. Проверка на вырожденном случае (устойчивость к дисбалансу)
    print("\n4. Проверка на устойчивость к вырождению (вставка 1..7):")
    avl2 = AVLTree()
    for i in range(1, 8):
        avl2.insert(i)
    print(f"Высота: {avl2.height()} (должна быть 3, а не 7)")
    print(f"Сбалансировано: {avl2.is_balanced()}")
    print(f"Обход: {avl2.inorder()}")

ТЕСТИРОВАНИЕ AVL ДЕРЕВА

1. Вставка элементов: 10, 20, 30, 40, 50, 25
Корень дерева: 30
Высота дерева: 3
Дерево сбалансировано? True
Отсортированный обход: [10, 20, 25, 30, 40, 50]

2. Поиск элементов:
Поиск 25: Найден
Поиск 100: Не найден

3. Удаление элементов:
Удаляем 40...
Высота: 3, Сбалансировано: True
Отсортированный обход: [10, 20, 25, 30, 50]
Удаляем 10 (корень)...
Высота: 3, Сбалансировано: True
Отсортированный обход: [20, 25, 30, 50]

4. Проверка на устойчивость к вырождению (вставка 1..7):
Высота: 3 (должна быть 3, а не 7)
Сбалансировано: True
Обход: [1, 2, 3, 4, 5, 6, 7]
